# Programming Tutorial 3 - Recurrent Neural Networks

```
Course: CSCI 4922/5922 Spring 2026, University of Colorado Boulder
TA: Everley Tseng
Email: Yu-Yun.Tseng@colorado.edu
* AI assistant was used in making this tutorial
```

## Overview

Sections:
- Recurrent layers: RNN, LSTM, and GRU
- Build recurrent models
- Character prediction
- Train recurrent model


Objectives:
- Learn how to use recurrent layers
- Learn about the loss function and data processing for multi-class classification
- Learn how to train a simple character prediction model
- Compare the RNN, LSTM, and GRU models

## Recurrent Layers

Similar to having `nn.Linear` for linear layers, PyTorch also has  modules for recurrent layers such as `nn.RNN`, `nn.LSTM`, and `nn.GRU`. In this tutorial, we will build three simple recurrent models using RNN, LSTM, and GRU layers, respectively, to build recurrent models. Before we start, read the documentation pages below to familiarize yourself with these layers:
- RNN: [`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)
- LSTM: [`nn.LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
- GRU: [`nn.GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)


**[Take a Pause Here]** These are long documentation pages. What information am I looking for when exploring a new layer type?

The tips are below:

1. Look for the class setup. Identify the arguments and what they stand for.
2. Look for the layer computation. Understand the inputs, outputs, and layer parameters.
3. Look for the `forward` function. Identify what each variable stands for and what to pass in and out of the `forward` function.

You can revisit the [Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer and look for these items. Below, we'll do a deep breakdown of the RNN layer as a demonstration.

### Example: RNN

The major difference in recurrent layers from the layers we have used is that the hidden states are passed forward over time steps.

By reading the page [`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html), we see that the layer takes in **2 elements** and returns **2 elements**. This aligns with what we learned about an RNN unit in the lecture, where there is a hidden state $h_t$ that passes over time steps. Following this format, we can structure the forward pass to be:
```
out, h2 = rnn_layer(in, h1)
```
The shape of these elements will be:
- Input `in`: `(batch_size, seq_length, feature_size)`
- Hidden states `h1` and `h2`: `(1, batch_size, hidden_size)`
- Output `out`: `(batch_size, seq_length, hidden_size)`

Both `in` and `out` accumulate over time steps (i.e., with the `seq_length` dimension), and the hidden state changes over time. Here, `batch_size` refers to the number of samples sent into the layer in one pass.


The setup for [`nn.LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) and [`nn.GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) are similar to `nn.RNN`. Please read the pages and familiarize yourself with these two classes.

### Model Summary

Recall that we used the tool `torchsummary` to visualize the model architecture when building a model with linear layers. Today, we will use the package `torchinfo` instead of `torchsummary`. As `torchsummary` cannot summarize complex data structures such as recurrent layers, you might run into errors if you use `torchsummary` to visualize an architecture that contains recurrent layers.

Install the package:

In [4]:
!pip install torchinfo

## Recurrent Models

Now, let's build some recurrent models!

In [5]:
import torch
import torch.nn as nn
from torchinfo import summary

### RNN Model

First, let's practice building a RNN classification model with one recurrent layer followed by a fully-connected layer.

In the **first time step, there is no prior hidden state**. We will initialize a hidden state for it (which is usually zeros). For the RNN layer output, we take the intermediate feature from the **final time step** istead of the entire sequence as we are only predicting the one upcoming character.



**nn.RNN(batch_first=True)**

In the RNN layer, `batch_size` is a default dimension, which refers to the number of samples being passed into the layer. In this case, we are sending one sample at a time.

When using `batch_first=True`, the shape of the input is expected to be `(batch_size, sequence, feature)`; when using `batch_first=False`, the shape of the input is expected to be `(sequence, batch_size, feature)`. Our data will be sent into the model sample by sample in the shape `(1, sequence, feature)`, so let's assign `batch_first=True` for the rnn layer.

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()

        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, hidden=None):
        # Pass in zeros when there is no prior hidden state
        if hidden is None:
            hidden = torch.zeros(1, x.size(0), self.hidden_size)

        # RNN layer
        rnn_out, hidden = self.rnn(x, hidden)

        # Fully-connected layer
        output = self.fc(rnn_out[:, -1, :]) # Take the last time step's output

        # Apply softmax for classification
        output = self.softmax(output)
        return output, hidden

In [ ]:
model = SimpleRNN(input_size=58, hidden_size=256, output_size=58)
summary(model, input_size=(1, 1, 58))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleRNN                                [1, 58]                   --
├─RNN: 1-1                               [1, 1, 256]               80,896
├─Linear: 1-2                            [1, 58]                   14,906
├─LogSoftmax: 1-3                        [1, 58]                   --
Total params: 95,802
Trainable params: 95,802
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.10
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.38
Estimated Total Size (MB): 0.39

In [ ]:
model = SimpleRNN(input_size=48, hidden_size=512, output_size=100)
model.rnn.weight_ih_l0.shape

torch.Size([512, 48])

In [ ]:
xb = torch.rand(1,256,48)
yb = torch.tensor([4])
rnn = nn.RNN(48, 512)
output, h_n = rnn(xb)


[('weight_ih_l0',
  Parameter containing:
  tensor([[ 0.0416,  0.0073, -0.0056,  ..., -0.0261, -0.0257, -0.0039],
          [ 0.0244,  0.0199,  0.0163,  ..., -0.0065, -0.0115, -0.0297],
          [ 0.0071, -0.0182, -0.0257,  ..., -0.0135,  0.0224,  0.0146],
          ...,
          [-0.0300, -0.0055, -0.0174,  ..., -0.0191, -0.0151,  0.0251],
          [ 0.0417, -0.0373, -0.0110,  ...,  0.0436,  0.0113,  0.0279],
          [ 0.0113, -0.0233, -0.0087,  ...,  0.0090,  0.0388,  0.0092]],
         requires_grad=True)),
 ('weight_hh_l0',
  Parameter containing:
  tensor([[ 0.0254, -0.0243,  0.0402,  ..., -0.0095,  0.0432, -0.0261],
          [-0.0250, -0.0135, -0.0117,  ..., -0.0266, -0.0300, -0.0232],
          [-0.0132, -0.0346,  0.0033,  ...,  0.0385,  0.0155, -0.0222],
          ...,
          [-0.0206,  0.0436,  0.0424,  ..., -0.0040,  0.0411,  0.0053],
          [-0.0387,  0.0232,  0.0054,  ..., -0.0361,  0.0036,  0.0059],
          [ 0.0125,  0.0282, -0.0364,  ...,  0.0405, -0.0351, 

### LSTM

Now, similar the the RNN model, we will build an LSTM model with one LSTM layer followed by a linear layer. As you can find in the page [`nn.LSTM`](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html), LSTM layer has two hidden states, $h_t$ and $C_t$, which is passed in the layer as a tuple:
```
out, (h2, C2) = rnn_layer(in, (h1, C1))
```
Let's try adopting the structure of RNN model and adjust the hidden states to include both `C` and `h` in LSTM.

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, hidden=None):
        if hidden is None:
            hidden = torch.zeros(1, x.size(0), self.hidden_size)  # Hidden state
            cell = torch.zeros(1, x.size(0), self.hidden_size)  # Cell state
            hidden = (hidden, cell)  # LSTM requires a tuple of (hidden_state, cell_state)

        lstm_out, hidden = self.lstm(x, hidden)
        output = self.fc(lstm_out[:, -1, :])
        output = self.softmax(output)
        return output, hidden

In [ ]:
model = SimpleLSTM(input_size=58, hidden_size=256, output_size=58)
summary(model, input_size=(1, 1, 58))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleLSTM                               [1, 58]                   --
├─LSTM: 1-1                              [1, 1, 256]               323,584
├─Linear: 1-2                            [1, 58]                   14,906
├─LogSoftmax: 1-3                        [1, 58]                   --
Total params: 338,490
Trainable params: 338,490
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.34
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 1.35
Estimated Total Size (MB): 1.36

### GRU

Finally, let's use a [`nn.GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) for the recurrent model.

In [2]:
import torch
import torch.nn as nn

class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleGRU, self).__init__()
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x, hidden=None):
        if hidden is None:
            hidden = torch.zeros(1, x.size(0), self.hidden_size)  # GRU only has hidden state

        gru_out, hidden = self.gru(x, hidden)
        output = self.fc(gru_out[:, -1, :])  # Take the last time step's output
        output = self.softmax(output)
        return output, hidden

In [6]:
model = SimpleGRU(input_size=58, hidden_size=256, output_size=58)
summary(model, input_size=(1, 1, 58))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleGRU                                [1, 58]                   --
├─GRU: 1-1                               [1, 1, 256]               242,688
├─Linear: 1-2                            [1, 58]                   14,906
├─LogSoftmax: 1-3                        [1, 58]                   --
Total params: 257,594
Trainable params: 257,594
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.26
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 1.03
Estimated Total Size (MB): 1.03

## Character Prediction Task

Today, we'll be training these recurrent models on **character prediction**.

### Definition

***What is character prediction?*** Given a string "How are yo", predict the next character: "u". The input string is a sequential data, where there is a time-step correlation in:
```
H -> o -> w -> space -> a -> r -> e -> space -> y -> o -> u
```
Therefore, we will be passing the input **character by character** into the model to predict the most likely upcoming character.

### Data Formatting Logistics

Let's prepare the dataset for training a **character prediction** model. We start with breaking down this task: Given $n$ characters, predict the ${n+1}^{th}$ character.

Character prediction can be considered as a multi-class classification problem, where the classes are the selection of characters, `a`, `b`, `c`, etc. In this setup, the given $n$ characters is a sequence of classes (i.e., characters), and the ${n+1}^{th}$ prediction will be a **multi-class classification prediction**.

Below, we'll walk through a simple example to learn about the data processing steps.


Assume that we have a corpus that has 21 characters in total, **"Deep Learning is fun."**

- **Step 1:** Convert all characters to lowercase. This means that we will consider the uppercase (i.e., `D`) and the lowercase (i.e., `d`) characters to be the same class.
  ```
  "Deep Learning is fun." -> "deep learning is fun."
  ```
- **Step 2:** Calculate the number of unique characters in the corpus (i.e., 14)
  ```
  "d", "e", "p", " ", "l", "a", "r", "n", "i", "g", "s", "f", "u", "."
  ```
- **Step 3:** Use one-hot encoding to encode each character. For instance, our dictionary size is 14, so the length of each character encoding is 14.
  ```
  "d": [1 0 0 0 0 0 0 0 0 0 0 0 0 0]
  "e": [0 1 0 0 0 0 0 0 0 0 0 0 0 0]
  "p": [0 0 1 0 0 0 0 0 0 0 0 0 0 0]
  " ": [0 0 0 1 0 0 0 0 0 0 0 0 0 0]
  "l": [0 0 0 0 1 0 0 0 0 0 0 0 0 0]
  ...
  ".": [0 0 0 0 0 0 0 0 0 0 0 0 0 1]
  ```
- **Step 4:** Create training data (`x` and `y`) using a sliding window. Determine a window size $n$, e.g., $n=5$. Determine a window step $s$, e.g., $s=1$. The smaller the step is, the more samples we have.
  ```
  Corpus: "deep learning is fun."
  sample 1: x=[deep ], y=[l]
  sample 2: x=[eep l], y=[e]
  sample 3: x=[ep le], y=[a]
  sample 4: x=[p lea], y=[r]
  sample 5: x=[ lear], y=[n]
  sample 6: x=[learn], y=[i]
  ...
  ```

- **Step 5:** Using the one-hot encoding dictionary created in **step 3**, convert all training data to the encoded style. Take sample 1 as an example:
  ```
  sample 1 x:
  "deep " is converted to a 5 x 14 embedding
  [
    [1 0 0 0 0 0 0 0 0 0 0 0 0 0],
    [0 1 0 0 0 0 0 0 0 0 0 0 0 0],
    [0 1 0 0 0 0 0 0 0 0 0 0 0 0],
    [0 0 1 0 0 0 0 0 0 0 0 0 0 0],
    [0 0 0 1 0 0 0 0 0 0 0 0 0 0]
  ]
  sample 1 y:
  "l" is converted to a 1 x 14 embedding
  [0 0 0 0 1 0 0 0 0 0 0 0 0 0]
  ```

Finally, each training sample will have:
- `x` with the shape `(seq_length, feature_size)`
- `y` with the shape `(feature_size)`

### Load and Process Dataset

1. Load data from the given [url](https://s3.amazonaws.com/text-datasets/nietzsche.txt).
2. Use the string `full_text` to store the entire corpus.
3. You will find that there are 600901 characters in total.

In [ ]:
import requests

text_url = 'https://s3.amazonaws.com/text-datasets/nietzsche.txt'
response = requests.get(text_url)
response.raise_for_status()
lines = response.text.splitlines()
full_text = ' '.join(lines)
print('Total # of characters in corpus:', len(full_text))
print('Print first 500 characters:', full_text[:500])

Total # of characters in corpus: 600901
Print first 500 characters: PREFACE   SUPPOSING that Truth is a woman--what then? Is there not ground for suspecting that all philosophers, in so far as they have been dogmatists, have failed to understand women--that the terrible seriousness and clumsy importunity with which they have usually paid their addresses to Truth, have been unskilled and unseemly methods for winning a woman? Certainly she has never allowed herself to be won; and at present every kind of dogma stands with sad and discouraged mien--IF, indeed, it s


Now, let's start processing our data!

In [ ]:
# Convert to lowercases
full_text = full_text.lower()

# Create a character dictionary
char_dict = sorted(set(full_text))

print('{} characters'.format(len(char_dict)))

58 characters


There are $58$ characters in this corpus, so our one-hot encoding features will be the size of $58$.

In [ ]:
# Create a dictionary for one-hot encoding
# key: character, value: encoding
encoder = {}

for ind, char in enumerate(char_dict):
    tensor = torch.zeros(len(char_dict)) # create an all-zero array
    tensor[ind] = 1                      # flag the index of the charactre as 1
    encoder[char] = tensor               # save to the encoder dict

char_ind_map = {ind:char for ind, (char, _) in enumerate(encoder.items())}

Set up a proper window size and a proper step size to generate samples for training.

In [ ]:
window_size = 40
window_step = 10

x_train = []
y_train_one_hot = []

for i in range(0, len(full_text)-window_size, window_step):
    # Sample x and y from corpus
    x = full_text[i:i+window_size]
    y = full_text[i+window_size]

    # Convert characters to encoding
    x_encoded = torch.stack([encoder[c] for c in x])
    y_encoded = encoder[y]
    x_train.append(x_encoded)
    y_train_one_hot.append(y_encoded)

x_train = torch.stack(x_train)
y_train_one_hot = torch.stack(y_train_one_hot)

print('Input shape: {}. Output shape: {}'.format(x_train.shape, y_train_one_hot.shape))
print('Sample x shape:', x_train[0].shape)
print('Sample y shape:', y_train_one_hot[0].shape)
print('Sample y:', y_train_one_hot[0])

Input shape: torch.Size([60087, 40, 58]). Output shape: torch.Size([60087, 58])
Sample x shape: torch.Size([40, 58])
Sample y shape: torch.Size([58])
Sample y: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.])


Due to time constraints, let's cut the samples at 6000. You are welcome to increase the sample size or use the full set!

In [ ]:
x_train = x_train[:6000,:,:]
y_train_one_hot = y_train_one_hot[:6000]

## Model Training

The common components in multi-class classification are introduced below:

- **Softmax**
  
  Softmax can convert the output into probabilities, and this calculation is available in this class [`nn.Softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html).

- **Cross-Entropy Loss**

  For this multi-class classification task, we'll be using cross-entropy loss: [`nn.CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

  In PyTorch, the `nn.CrossEntropyLoss` class encompasses the softmax layer, so we don't need to implement `nn.Softmax` when training the model.

  In PyTorch, `nn.CrossEntropyLoss` automatically converts the classes into one-hot encoding. Therefore, we don't need to format `y` in the one-hot encoding style. Instead, we can convert the encoded `y_train_one_hot` back into the logit style by using [`argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html).


In [ ]:
y_train = y_train_one_hot.argmax(dim=1)
print('y_train shape:', y_train.shape)
print('Sample y:', y_train[0])

y_train shape: torch.Size([6000])
Sample y: tensor(39)


### RNN Model Training [Running Takes ~10 Minutes]

Let's write a function for training that will fit all of our models (i.e., RNN, LSTM, and GRU models). During training, save the training loss and training time for comparison.


In [ ]:
import time
import numpy as np
import torch.optim as optim

def train_model(model, x_train, y_train, lr=0.001, n_epochs=10, print_every=1):
    model.train()

    n_train = x_train.size(0)                         # number of samples
    criterion = nn.CrossEntropyLoss()                 # loss function
    optimizer = optim.SGD(model.parameters(), lr=lr)  # SGD optimizer
    losses = []                                       # save loss values
    epoch_times = []                                  # save time to train an epoch

    for epoch in range(n_epochs):
        total_loss = 0.0
        start_time = time.time()

        # randomize indices for training samples
        indices = torch.randperm(n_train)

        # loop over individual examples
        for ind in indices:

            # get sample
            x = x_train[ind:ind+1,:,:]
            y = y_train[ind:ind+1]

            # forward and backward
            optimizer.zero_grad()
            output, hidden = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            # save loss
            total_loss += loss.item()

        end_time = time.time()
        avg_loss = total_loss / n_train
        losses.append(avg_loss)
        epoch_times.append(end_time - start_time)

        # print results every n (e.g., 1) epochs
        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.4f}")

    avg_time = float(np.mean(epoch_times)) if len(epoch_times) > 0 else 0.0
    print(f"Training complete! Average computation time is {avg_time:.2f}s per epoch.")
    return model, losses

Create a `SimpleRNN` model and pass it into the training function. The training might take a while.

In [ ]:
input_size = 58
hidden_size = 256
output_size = 58

rnn_model = SimpleRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size)
rnn_model, rnn_losses = train_model(rnn_model, x_train, y_train, n_epochs=10)

Epoch 1/10, Loss: 3.2283
Epoch 2/10, Loss: 2.9907
Epoch 3/10, Loss: 2.9696
Epoch 4/10, Loss: 2.9444
Epoch 5/10, Loss: 2.9158
Epoch 6/10, Loss: 2.8726
Epoch 7/10, Loss: 2.8237
Epoch 8/10, Loss: 2.7764
Epoch 9/10, Loss: 2.7317
Epoch 10/10, Loss: 2.6925
Training complete! Average computation time is 44.22s per epoch.


### RNN Model Prediction

1. **Single Prediction**

   Now, let's use the trained model to predict a character. Use a sequence as input to predict the next character:

In [ ]:
input_text = 'philosop' # change this to any character sequence

with torch.no_grad():
    x_encoded = torch.stack([encoder[c] for c in input_text]).unsqueeze(0)
    hidden = torch.zeros(1, 1, hidden_size) # create zeros for initial hidden state
    output, hidden = rnn_model(x_encoded, hidden)
    predicted_index = output.argmax(dim=1).item()
    print(output.shape, predicted_index)

print('Predicted character:', char_ind_map[predicted_index])
print('Full string:', input_text + char_ind_map[predicted_index])

torch.Size([1, 58]) 30
Predicted character: e
Full string: philosope


2. **Continuous Prediction**

   To generate a longer sequence, we can use the prediction recurrently to generate continuous results.
   
   Let's take a random 40-character sample in the corpus and generate the next 50 characters. As the model generates a new character, we will update the `input_text` like a moving window to make the next prediction.

In [ ]:
def generate_characters(model, hidden, input_text, input_size=40, n_char=50):
    output_text = input_text
    with torch.no_grad():
        for _ in range(n_char):
            x_encoded = torch.stack([encoder[c] for c in input_text]).unsqueeze(0) # encode input text
            output, hidden = model(x_encoded, hidden)
            predicted_index = output.argmax(dim=1).item()
            output_text = output_text + char_ind_map[predicted_index] # convert the predicted index to character
            input_text = output_text[-input_size:] # update the input text
    return output_text

In [ ]:
start_index = 103 # a random index
input_text = full_text[start_index:start_index+40]
print('* Input text:\n', input_text)

hidden = torch.zeros(1, 1, hidden_size) # create zeros for initial hidden state
output_text = generate_characters(rnn_model, hidden, input_text)
print('* Prediction:\n', output_text)

* Input text:
 sophers, in so far as they have been dog
* Prediction:
 sophers, in so far as they have been dog the the the the the the the the the the the the t


This model hasn't converged well on this model. We encourage you to **increase the number of epochs** and train the model longer for a better character prediction!

When the model is in a better shape for character prediction, you might notice that the first few generated characters make more sense, and the further into the predictions the worse it gets. This is due to the **recurrently updating input text using the predictions**. Once there is **error in the predictions**, it can cause a snowballing effect on the future predictions. This phenomenon is known as error accumulation, where incorrect outputs are fed back into the model as input, reinforcing deviations from meaningful text.

### LSTM Model Training



Below is the sample code to train the LSTM model. You can increase the number of epochs and compare it results to the RNN model results.

In [ ]:
lstm_model = SimpleLSTM(input_size=input_size, hidden_size=hidden_size, output_size=output_size)
lstm_model, lstm_losses = train_model(lstm_model, x_train, y_train, n_epochs=10)

Epoch 1/10, Loss: 3.8513
Epoch 2/10, Loss: 3.2997
Epoch 3/10, Loss: 3.0414
Epoch 4/10, Loss: 2.9951
Epoch 5/10, Loss: 2.9824
Epoch 6/10, Loss: 2.9775
Epoch 7/10, Loss: 2.9740
Epoch 8/10, Loss: 2.9707
Epoch 9/10, Loss: 2.9686
Epoch 10/10, Loss: 2.9671
Training complete! Average computation time is 43.63s per epoch.


### GRU Model Training

Below is the sample code to train the GRU model. You can increase the number of epochs and compare it results to the RNN model results.

In [ ]:
gru_model = SimpleGRU(input_size=input_size, hidden_size=hidden_size, output_size=output_size)
gru_model, gru_losses = train_model(gru_model, x_train, y_train, n_epochs=1)

Epoch 1/1, Loss: 3.6621
Training complete! Average computation time is 139.68s per epoch.


## Review

Recurrent architectures are usually more time-consuming to train. To perform experiments, you can use shorter sequences (i.e., set up a smaller sequence window size).

You can complete the following tasks to practice coding recurrent architectures:
1. Increase the number of epochs for the RNN model to see if the character prediction results improve.
2. Create a deeper RNN model. Train and compare its training time and results to the 1-layer RNN model.

For any questions and discussions regarding this tutorial, attend [TA office hours](https://docs.google.com/spreadsheets/d/1abWD9DJqjEGrCdr8VbZ3aiOhx4vpT1y0-LOoLWgPwZM/edit?usp=sharing) or create a post on [Piazza](https://piazza.com/colorado/spring2026/csci49225922/home) :) See you in the next tutorial!

\- Everley